In [1]:
import pyspark.sql, pyspark.sql.functions
from pyspark.sql import SparkSession

# Стартиране на SparkSession 
spark = (
    SparkSession.builder
    .appName("Split validation example")
    .master("local[*]")
    .getOrCreate()
)
print("SparkSession стартиран успешно")
print("Spark версия:", spark.version)

SparkSession стартиран успешно
Spark версия: 4.0.1


In [4]:
# Зареждане на CSV в DataFrame
df = spark.read.csv('students-online.csv', header=True, inferSchema=True)
# Премахваме ID, ако не участва в модела
df = df.drop("ID")

In [5]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

# Подготовка на характеристиките
feature_cols = ["Age","PlatformHours","OutPlatformHours"]
# Обединяване на характеристиките във вектор
assembler = VectorAssembler(inputCols=feature_cols,outputCol="features")
data = assembler.transform(df)

# Избор на колони
data = data.select("features", "Satisfaction")

# Разделяне на данните: 70% обучаващ набор и 30% тестов набор
train_data, test_data = data.randomSplit([0.7, 0.3], seed=42)

# Създаване на модел
lr = LinearRegression(featuresCol="features", labelCol="Satisfaction")

# Обучение
model = lr.fit(train_data)

# Прогнозиране
predictions = model.transform(test_data)

# Оценка на модела
evaluator = RegressionEvaluator(
    labelCol="Satisfaction",
    predictionCol="prediction",
    metricName="rmse"
)

rmse = evaluator.evaluate(predictions)

print("RMSE:", rmse)

RMSE: 0.45103053944582455
